# 08 · Ablations

Each run changes one thing in CFFM-Net: **gating off** (C1, PH2), a **gated convolution** of matched compute instead of the scan (C4, PH4), and optionally **no stride-4 head** (C2).

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot`, raw LLVIP and M3FD (or the output of 02) |
| **Expected runtime** | about 4 to 5 hours on 2 x T4 (about 6 with the optional run) |
| **Produces** | checkpoints in `runs/<run>/`, full-set metrics in `runs/eval/<run>/metrics.json` |

In [ ]:
# Same setup cell in every notebook: find the project, install it, check the GPU.
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        # Kaggle mounts datasets at different depths under /kaggle/input, so search five levels.
        hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
                if (Path(p).parent / "src" / "cffm").is_dir()]
        # Earlier notebooks' outputs hold stale copies of the code, so prefer the uploaded dataset.
        hits.sort(key=lambda h: any((h.parent / d).exists() for d in ("runs", "data")))
        zips = [Path(p) for k in range(1, 5) for p in glob.glob("/kaggle/input" + "/*" * k + ".zip")
                if "cffm" in Path(p).name]
        if hits:
            shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        elif zips:                       # the zip was attached but not unpacked by Kaggle
            shutil.unpack_archive(str(zips[0]), "/kaggle/working")   # it holds one folder, cffm-net-pilot/
        assert PROJECT.exists(), "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
        for p in [PROJECT, *PROJECT.rglob("*")]:     # inputs are read-only; our copy must be writable
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle", "pytest"], check=True)   # cloudpickle: two-GPU launcher
    # Editable install, so DataLoader and DDP worker processes can import cffm too.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)   # must be inside PROJECT
env = cenv.setup()                       # also moves into the project folder
plan = pipeline.load_plan(env.project)   # configs/pilot.yaml

## The runs

Straight from `configs/pilot.yaml`. Batch is the total over both GPUs.

In [ ]:
INCLUDE_OPTIONAL = False    # True also trains optional runs (the no-stride-4-head ablation)
specs = [s for s in pipeline.runs_for(plan, "08") if INCLUDE_OPTIONAL or not s.get("optional")]

import pandas as pd
cols = ["name", "model", "data", "epochs", "imgsz", "batch"]
pd.DataFrame(specs)[cols]

## Train and evaluate

In [ ]:
# Finished runs are reused and unfinished ones resume, so just rerun if the Kaggle session dies.
results = [pipeline.train_from_spec(s, env) for s in specs]

## Learning curves

In [ ]:
# Val mAP still rising at the end is expected; val mAP falling with the loss would mean overfitting.
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for s in specs:
    f = env.runs / s["name"] / "results.csv"
    if not f.exists():
        continue
    df = pd.read_csv(f)
    df.columns = [c.strip() for c in df.columns]
    axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], label=s["name"])
    axes[1].plot(df["epoch"], df["metrics/mAP50(B)"], label=s["name"])
    loss_cols = [c for c in df.columns if c.startswith("train/") and c.endswith("loss")]
    axes[2].plot(df["epoch"], df[loss_cols].sum(axis=1), label=s["name"])
for a, t in zip(axes, ["val mAP50-95 (quarter val set)", "val mAP50", "train loss (sum)"]):
    a.set_title(t); a.set_xlabel("epoch"); a.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Results on the full validation set

In [ ]:
# AP_vt / AP_t / AP_s are the AI-TOD size bands (<8, 8-16, 16-32 px).
import pandas as pd
keys = ["mAP50-95(B)", "mAP50(B)", "mAP_small(B)", "AP_vt(B)", "AP_t(B)", "AP_s(B)", "AP_m(B)", "AP_l(B)"]
rows = [{"run": r["run"], **{k.replace("(B)", ""): r["metrics"].get(k) for k in keys}} for r in results]
pd.DataFrame(rows).set_index("run").round(4)

In [ ]:
# /kaggle/working becomes the saved output, and the data is cheap to rebuild, so keep only the runs.
if env.platform == "kaggle":
    shutil.rmtree(env.data, ignore_errors=True)

### Keep the results

On Kaggle, click **Save Version → Save & Run All (Commit)** so `runs/` becomes an output that notebooks **09**
and **15** can attach (*Add Input → Your Work → this notebook*). Without a saved version the checkpoints vanish.